# Riksdagens tjänstgörande personer: bildluckor i Wikidata

* Denna [notebook](https://github.com/salgo60/Wikidata_riksdagen-corpus/blob/main/Notebook/Riksdagen_wikidata_bildkontroll.ipynb)
* Bilder på riksdagsledamöter diskuteras på [Wikipedia:Bybrunnen](https://sv.wikipedia.org/wiki/Wikipedia:Bybrunnen#Bilder_p%C3%A5_riksdagsledam%C3%B6ter)

## Versioner

### Version 2
* Visar även personer som har bild.
* Kan filtrera på personer som saknar artikel på svenska Wikipedia.

### Version 3
* Commons-bilden visas och kan jämföras med Riksdagens porträtt, med licens och fotograf/upphovsperson.
* Färdigskrivna mejlutkast till ledamöterna med en önskan om porträttbilder under fri licens.

### Version 4
* Utfällbar statistikdel med antal och andel personer som har fri Commons-bild, per parti och län.

### Version 5
* Stapeldiagram över bildläget per parti.
* Interaktiv Sverigekarta över de 21 länen: växla mellan antal och andel som saknar fri bild, välj ett län och jämför med länsrankningen.
* Karta och graf är inbäddade i HTML-filen och fungerar utan nätverk.

### Version 6
* Kontrollerar `{{illustrationsbehov-svensk politiker}}` i svenska artiklar för identifierarmatchade personer som saknar P18.
* Filtret **Saknad mall (utan P18)** visar bara bekräftat saknade mallar. Misslyckade hämtningar markeras som okända.
* Wikipedia-API:s mallista används; kontrollen inkluderar mallens omdirigeringar. Kontrolltid visas på personkortet. Ingen artikel redigeras.

### Version 7
* Sökning och samtliga filter sparas i URL:en som ett fragment, exempelvis `#party=M&illustration=saknas`.
* En delad länk återställer samma urval när rapporten öppnas. Funktionen fungerar även för lokala HTML-filer.

## Funktion

Källa: [Riksdagens personlista](https://data.riksdagen.se/personlista/?iid=&fnamn=&enamn=&f_ar=&kn=&parti=&valkrets=&org=).

Matcha `sourceid` → **P8388** och kontrollera **P18**. Kontrollera även `intressent_id` → **P1214** vid utebliven GUID-matchning. Namnsökning på exakta svenska/engelska etiketter ger möjliga kandidater; namnmatchning godkänns aldrig automatiskt.

Notebooken innehåller en datasnapshot från **2026-10-07** och fungerar utan nätverk med `REFRESH=False`. Ändra till `True` för nya källdata. Riksdagens lista omfattar även tjänstgörande ersättare.

**Utdata:** sökbar HTML-rapport, hela jämförelsen i CSV/JSON, statistiktabeller och sammanfattning. Inga ändringar görs i Wikidata.

**Wikipedia-länkar:** tillgängliga språk, svenska och engelska först. Filtret gäller svenska Wikipedia: finns, saknas eller okänt. Namnkandidater är inte verifierade identiteter.

**HTTP 429:** User-Agent innehåller `salgo60@msn.com`. Minst 65 sekunder mellan WDQS-frågor; Retry-After respekteras och HTTP 429/503 ger högst fyra försök. Väntetid syns i utdata. Stop/Interrupt avbryter. Misslyckade hämtningar skriver inte över tidigare cachefiler.

Den senaste fullständiga uppdateringen av person- och Wikidatauppgifter slutfördes **7 oktober 2026 kl. 16.23 svensk tid**. Riksdagens systemdatum: 2026-10-07 16:20:41. Alla 349 matchar via P8388; 167 har P18 och 182 saknar P18. Detta är snapshotens resultat, inte fasta förväntade antal för framtida körningar.

**Commons-bilder:** alla personer visas. P18-bilder har licenslänk och fotograf/upphovsperson från Artist-fältet. Credit, Attribution och restriktioner visas när de finns. Uppladdaren används aldrig som ersättning för fotograf. Saknade uppgifter markeras. Metadata hämtas i små sekventiella batcher och sparas för offlinekörning. P18 är inte nödvändigtvis bilden som en Wikipedia-artikel använder.

**Mejlutkast:** officiell e-post hämtas från samma personpost som person-ID. Personliga mailto-utkast har namn, ämne, mejltext och länkar till personens Wikipedia-artiklar på alla tillgängliga språk, med svenska och engelska först. `EMAIL_SENDER` anger signaturen, nu Magnus Sälgö. Texten kan visas och kopieras. Ingen e-post skickas automatiskt.

**Statistik:** varje person räknas högst en gång. Län härleds från valkrets: Stockholm stad/län, Malmö/Skåne och Göteborg/Västra Götaland grupperas. Statistiken gäller hela listan, inte aktiva filter. Okända licenser redovisas separat. Kartans mått saknar fri bild inkluderar okänd licens. Kartgeometrin är förenklad och kommer från [okfse/sweden-geojson](https://github.com/okfse/sweden-geojson); län avser valkretsen, inte bostadsorten.


In [1]:
# REFRESH=False använder inbyggda data, inklusive Commons metadata.
# REFRESH=True hämtar nya uppgifter. WDQS-frågorna skickas med minst 65 sekunders mellanrum.
# Python 3; inga externa paket behövs för jämförelse och HTML-export.
from pathlib import Path
import json, base64, zlib
REFRESH = True
OUTPUT_DIR = Path('riksdagen_audit_20261008_v10')
OUTPUT_DIR.mkdir(exist_ok=True)
SNAPSHOT = '...'
for filename, data in json.loads(zlib.decompress(base64.b64decode(SNAPSHOT))).items():
    path = OUTPUT_DIR / filename
    if not path.exists():
        path.write_text(json.dumps(data, ensure_ascii=False), encoding='utf-8')
print('Utdata:', OUTPUT_DIR.resolve())


Utdata: /Users/salgo/Documents/GitHub/Wikidata_riksdagen-corpus/Notebook/riksdagen_audit_20261008_v10


## Hämtning, jämförelse och rapportfunktioner
SPARQL använder truthy-värden: föråldrade utsagor räknas inte och bästa rank styr vilka värden som syns. API-fel avbryter körningen. Wikidatas index kan släpa efter senaste ändringar.

In [2]:
"""Riksdagen → Wikidata: identifierarmatchning och bildkontroll. Python 3, inga externa paket krävs."""
import csv
import html
import json
import time
import hashlib
import re
import math
from html.parser import HTMLParser
from collections import defaultdict, Counter
from datetime import datetime, timezone
from pathlib import Path
from urllib.parse import quote, urlencode, unquote

from urllib.request import Request, urlopen
from urllib.error import HTTPError
from email.utils import parsedate_to_datetime

SOURCE = 'https://data.riksdagen.se/personlista/?iid=&fnamn=&enamn=&f_ar=&kn=&parti=&valkrets=&org=&utformat=json'
ENDPOINT = 'https://query.wikidata.org/sparql'
Q8388 = 'SELECT ?item ?id ?image WHERE {?item wdt:P8388 ?id. OPTIONAL {?item wdt:P18 ?image}}'
Q1214 = 'SELECT ?item ?id WHERE { ?item wdt:P1214 ?id . }'
HEADERS = {'User-Agent': 'RiksdagenImageAudit/1.1 (User:Salgo60; mailto:salgo60@msn.com)', 'Accept': 'application/json'}
WDQS_MIN_INTERVAL = 65  # Begränsa oss till högst en WDQS-fråga per minut.
MAX_ATTEMPTS = 4
EMAIL_SENDER = '<ange ditt namn här>'
EMAIL_SUBJECT = 'Porträttbild till Wikipedia under fri licens'

def official_email(person):
    entries = person.get('personuppgift', {}).get('uppgift', [])
    if isinstance(entries, dict):
        entries = [entries]
    addresses = set()
    for entry in entries:
        if entry.get('typ') != 'eadress' and entry.get('kod') != 'Officiell e-postadress':
            continue
        # E-post hämtas från samma personpost; ingen adress konstrueras från namnet.
        if entry.get('intressent_id') and entry['intressent_id'] != person['intressent_id']:
            raise ValueError('E-postens person-ID stämmer inte med ledamoten.')
        values = entry.get('uppgift', [])
        if isinstance(values, str):
            values = [values]
        for value in values:
            address = value.replace('[på]', '@').strip()
            if re.fullmatch(r'[^\s@<>;,]+@[^\s@<>;,]+\.[^\s@<>;,]+', address):
                addresses.add(address)
    if len(addresses) > 1:
        raise ValueError('Flera officiella e-postadresser för '+person['sourceid']+'; välj mottagare manuellt.')
    return next(iter(addresses), '')

def email_body(name, wikipedia=()):
    articles = ('\n\nHär finns Wikipedia-artiklarna om dig:\n'+
                '\n'.join('- '+v['sprak']+': '+v['titel']+' – '+v['url'] for v in wikipedia)) if wikipedia else ''
    return f'''Hej {name},

Jag bidrar till Wikipedia och vill gärna att artikeln om dig ska kunna illustreras med en aktuell porträttbild. Skulle du kunna hjälpa oss genom att ladda upp en bild till Wikimedia Commons under en fri licens, exempelvis CC BY 4.0?{articles}

Det kräver att fotografen eller den som äger rättigheterna godkänner licensen. Fotografen anges vid användning, och bilden får återanvändas och bearbetas, även i kommersiella sammanhang.

Digg och PRV rekommenderar öppna licenser för offentlig information, som ett komplement till EU öppna data-direktiv från 2019 (https://eur-lex.europa.eu/legal-content/SV/TXT/?uri=CELEX%3A32019L1024). Riksdagens nuvarande bildvillkor anger däremot ingen sådan tydlig fri licens för ledamöternas porträtt trots begäran om det.

En bild från dig vore ett konkret steg framåt. Jag hjälper gärna till med uppladdningen och uppgifter om fotograf och licens.
Du får också gärna lyfta frågan inom riksdagen: varför inte publicera ledamöternas porträtt som ett samlat dataset med tydliga fria licenser?
Hjälp oss att komma vidare och göra våra folkvalda lättare att presentera på Wikipedia.

Vänliga hälsningar,
{EMAIL_SENDER}'''

def wait_before_request(seconds, message):
    if seconds > 0:
        print(message+' ('+str(round(seconds))+' sekunder).', flush=True)
        # Korta väntesteg gör att körningen kan avbrytas i Jupyter.
        deadline = time.monotonic() + seconds
        while time.monotonic() < deadline:
            time.sleep(min(1, max(0, deadline-time.monotonic())))

def retry_delay(error, attempt):
    value = error.headers.get('Retry-After') if error.headers else None
    delay = 0
    if value:
        try:
            delay = float(value)
        except ValueError:
            try:
                delay = parsedate_to_datetime(value).timestamp()-time.time()
            except (ValueError, TypeError, OverflowError):
                pass
    return max(delay, WDQS_MIN_INTERVAL * (2 ** attempt))

def fetch(url, path, refresh=False, params=None):
    path = Path(path)
    if not refresh and path.exists():
        return json.loads(path.read_text(encoding='utf-8'))
    if params:
        url += ('&' if '?' in url else '?') + urlencode(params)
    is_wdqs = url.startswith(ENDPOINT+'?')
    # Sparas mellan körningar/kärnomstarter så ett omedelbart återförsök inte
    # börjar med ännu en fråga inom samma minut.
    pacing_file = path.parent / '_wdqs_last_request.json'
    for attempt in range(MAX_ATTEMPTS):
        if is_wdqs:
            last = 0
            if pacing_file.exists():
                try:
                    last = float(json.loads(pacing_file.read_text())['timestamp'])
                except (ValueError, KeyError, TypeError):
                    pass
            wait_before_request(max(0, WDQS_MIN_INTERVAL-(time.time()-last)),
                                'Väntar för att respektera Wikidatas frågegräns')
            pacing_file.write_text(json.dumps({'timestamp': time.time()}))
        print('Hämtar '+path.name+' …', flush=True)
        try:
            # Övriga HTTP- och JSON-fel avbryter; inga falska saknade värden.
            with urlopen(Request(url, headers=HEADERS), timeout=90) as response:
                data = json.load(response)
            if 'error' in data:
                raise ValueError('API-fel; tidigare data behålls: '+str(data['error']))
            if is_wdqs and 'results' not in data:
                raise ValueError('Wikidata returnerade inget SPARQL-resultat; tidigare rapport behålls.')
            break
        except HTTPError as error:
            if error.code not in (429, 503) or attempt == MAX_ATTEMPTS-1:
                raise
            delay = retry_delay(error, attempt)
            print('HTTP '+str(error.code)+': tillfällig begränsning. Försök '+
                  str(attempt+2)+'/'+str(MAX_ATTEMPTS)+'. Tidigare data behålls.', flush=True)
            wait_before_request(delay, 'Väntar före återförsök')
    temporary = path.with_suffix(path.suffix+'.tmp')
    temporary.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding='utf-8')
    temporary.replace(path)
    return data

def sparql(query, path, refresh=False):
    return fetch(ENDPOINT, path, refresh, {'query': query, 'format': 'json'})['results']['bindings']

def index(bindings):
    result = defaultdict(lambda: defaultdict(set))
    for b in bindings:
        qid = b['item']['value'].rsplit('/', 1)[-1]
        images = result[b['id']['value'].strip().lower()][qid]
        if 'image' in b:
            images.add(b['image']['value'])
    return result

def plain_text(value):
    # Commons extmetadata innehåller HTML; visa text, inte opålitlig HTML.
    class TextParser(HTMLParser):
        def __init__(self):
            super().__init__(convert_charrefs=True)
            self.parts = []
        def handle_data(self, data):
            self.parts.append(data)
    parser = TextParser()
    parser.feed(value or '')
    return ' '.join(' '.join(parser.parts).split())

def commons_metadata(image_urls, out, refresh=False):
    filenames = sorted({unquote(url.rsplit('/', 1)[-1]).replace('_', ' ') for url in image_urls})
    result = {}
    for start in range(0, len(filenames), 5):
        titles = ['File:'+name for name in filenames[start:start+5]]
        key = hashlib.sha256('|'.join(titles).encode()).hexdigest()[:16]
        cache = out/('commons_batch_'+key+'.json')
        if refresh or not cache.exists():
            time.sleep(1)  # Små sekventiella batcher för extmetadata.
        data = fetch('https://commons.wikimedia.org/w/api.php', cache, refresh,
                     {'action': 'query', 'format': 'json', 'formatversion': '2',
                      'titles': '|'.join(titles), 'prop': 'imageinfo', 'iiprop': 'url|extmetadata',
                      'iiurlwidth': '192', 'iiextmetadatalanguage': 'sv',
                      'iiextmetadatafilter': 'Artist|Credit|Attribution|LicenseShortName|LicenseUrl|UsageTerms|Restrictions'})
        if 'error' in data or 'query' not in data:
            raise ValueError('Commons API-fel: '+str(data.get('error', data)))
        for page in data['query']['pages']:
            filename = page['title'].removeprefix('File:').replace('_', ' ')
            info = (page.get('imageinfo') or [{}])[0]
            meta = info.get('extmetadata', {})
            value = lambda name: plain_text(meta.get(name, {}).get('value', ''))
            result[filename] = {
                'filnamn': filename,
                'commons_url': info.get('descriptionurl', 'https://commons.wikimedia.org/wiki/File:'+quote(filename.replace(' ', '_'))),
                'bild_url': info.get('thumburl', info.get('url', '')),
                'licens': value('LicenseShortName') or value('UsageTerms') or 'Ej angiven i Commons',
                'licens_url': meta.get('LicenseUrl', {}).get('value', ''),
                'fotograf_upphovsperson': value('Artist') or 'Ej angiven i Commons',
                'credit': value('Credit'), 'attribution': value('Attribution'),
                'restriktioner': value('Restrictions'),
                'metadata_status': 'hämtad' if info else 'fil saknas/metadata saknas'}
    (out/'commons_metadata.json').write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding='utf-8')
    return result

def county_from_constituency(constituency):
    if constituency == 'Stockholms kommun':
        return 'Stockholms län'
    if constituency == 'Malmö kommun' or constituency.startswith('Skåne läns '):
        return 'Skåne län'
    if constituency == 'Göteborgs kommun' or constituency.startswith('Västra Götalands läns '):
        return 'Västra Götalands län'
    if constituency.endswith(' län'):
        return constituency
    return 'Ej mappad valkrets: '+constituency

def is_free_commons_license(image):
    license_name = image.get('licens', '').strip()
    # Identifierade fria licenser; okända namn räknas separat.
    # CC BY/CC BY-SA tillåter bearbetningar och kommersiell användning.
    return bool(re.fullmatch(r'CC BY(?:-SA)? [1234]\.0(?: [a-z]{2})?', license_name)) or license_name in {
        'CC0', 'Public domain', 'Copyrighted free use', 'European Parliament'}

def image_statistics(rows):
    result = {}
    for field in ('parti', 'lan'):
        groups = defaultdict(lambda: {'antal_personer': 0, 'med_fri_commons_bild': 0,
                                     'utan_commons_bild': 0, 'licens_okand': 0})
        for row in rows:
            group = groups[row[field] or 'Ej angivet']
            group['antal_personer'] += 1
            if row['fri_commons_bild']:
                group['med_fri_commons_bild'] += 1
            elif row['commons_bilder']:
                group['licens_okand'] += 1
            else:
                group['utan_commons_bild'] += 1
        table = []
        for name, counts in sorted(groups.items()):
            assert counts['antal_personer'] == sum(counts[k] for k in ('med_fri_commons_bild', 'utan_commons_bild', 'licens_okand'))
            table.append(dict(grupp=name, **counts,
                              andel_procent=round(100*counts['med_fri_commons_bild']/counts['antal_personer'], 1)))
        assert sum(group['antal_personer'] for group in table) == len(rows)
        result[field] = table
    return result

def party_chart(statistics):
    data = sorted(statistics['parti'], key=lambda v: v['utan_commons_bild'], reverse=True)
    maximum = max(v['antal_personer'] for v in data)
    width, height = 720, 60+len(data)*39
    parts = [f'<svg class="party-chart" viewBox="0 0 {width} {height}" role="img" aria-label="Bilder per parti: med fri Commons-bild, utan bild och okänd licens">']
    for index, v in enumerate(data):
        y = 20+index*39
        parts.append(f'<text x="12" y="{y+19}" font-size="15">'+html.escape(v['grupp'])+'</text>')
        x = 64.0
        for key, color, label in (('med_fri_commons_bild', '#287a9d', 'Med fri bild'), ('utan_commons_bild', '#d35a45', 'Utan bild'), ('licens_okand', '#dba634', 'Licens okänd')):
            count = v[key]
            bar_width = 500*count/maximum
            parts.append(f'<rect x="{x:.2f}" y="{y}" width="{bar_width:.2f}" height="27" fill="{color}"><title>'+html.escape(v['grupp']+': '+label+' '+str(count))+'</title></rect>')
            if count and bar_width > 26:
                parts.append(f'<text x="{x+bar_width/2:.2f}" y="{y+18}" text-anchor="middle" font-size="13" fill="white">{count}</text>')
            x += bar_width
        parts.append(f'<text x="{x+9:.2f}" y="{y+18}" font-size="13">{v["antal_personer"]} personer</text>')
    parts.append('</svg>')
    return '<section class="party-visual"><h2>Bildläge per parti</h2><p>Staplarna visar antal personer. Sorterat efter antal utan Commons-bild.</p>'+''.join(parts)+'<p class="chart-legend"><span style="color:#287a9d">■ Med fri bild</span> <span style="color:#d35a45">■ Utan bild</span> <span style="color:#a77b13">■ Licens okänd</span></p></section>'

def county_map(statistics, geometry):
    values = {v['grupp']:v for v in statistics['lan']}
    def polygons(feature):
        g = feature['geometry']
        return [g['coordinates']] if g['type']=='Polygon' else g['coordinates']
    def project(coordinate):
        lon, lat = coordinate[:2]
        return lon*math.cos(math.radians(62)), -lat
    points = [project(c) for f in geometry['features'] for p in polygons(f) for ring in p for c in ring]
    xmin, xmax = min(p[0] for p in points), max(p[0] for p in points)
    ymin, ymax = min(p[1] for p in points), max(p[1] for p in points)
    scale = min(320/(xmax-xmin), 590/(ymax-ymin))
    def xy(c):
        x,y=project(c)
        return 20+(x-xmin)*scale, 20+(y-ymin)*scale
    maximum = max(v['antal_personer']-v['med_fri_commons_bild'] for v in values.values())
    parts = ['<section class="county-visual"><h2>Var saknas flest fria bilder?</h2><p><label for="map-metric">Färga kartan efter: </label><select id="map-metric"><option value="count">Antal som saknar fri bild</option><option value="percent">Andel som saknar fri bild (%)</option></select></p><div class="county-map-layout"><div><svg id="county-map" viewBox="0 0 370 640" role="group" aria-label="Sveriges 21 län, färgade efter saknade fria Commons-bilder">']
    names = []
    for feature in geometry['features']:
        base = feature['properties']['name']
        matches = [name for name in values if name.removesuffix(' län') in (base, base+'s')]
        if len(matches)!=1:
            raise ValueError('Kartans län kunde inte matchas entydigt: '+base)
        name=matches[0]
        names.append(name)
        v=values[name];missing=v['antal_personer']-v['med_fri_commons_bild'];percent=100*missing/v['antal_personer']
        d=[]
        for polygon in polygons(feature):
            for ring in polygon:
                coords=[xy(c) for c in ring]
                if coords:
                    d.append('M'+' L'.join(f'{x:.2f},{y:.2f}' for x,y in coords)+' Z')
        label=f"{name}: {missing} av {v['antal_personer']} saknar fri bild ({percent:.1f} %). {v['med_fri_commons_bild']} har fri bild."
        parts.append('<path class="county-path" tabindex="0" role="button" fill-rule="evenodd" stroke="#fff" stroke-width="1.1" aria-label="'+html.escape(label,quote=True)+'" data-name="'+html.escape(name,quote=True)+'" data-count="'+str(missing)+'" data-percent="'+f'{percent:.2f}'+'" data-label="'+html.escape(label,quote=True)+'" d="'+html.escape(' '.join(d),quote=True)+'"><title>'+html.escape(label)+'</title></path>')
    assert set(names)==set(values), 'Alla statistikens län måste finnas på kartan'
    parts += ['</svg><div class="map-scale"><span>0</span><div></div><span id="map-max">'+str(maximum)+'</span></div><p id="map-caption">Antal personer som saknar fri Commons-bild</p></div><div><p id="county-selection" role="status">Klicka på ett län eller använd Tab och Enter för att se detaljer.</p><h3>Län med flest saknade bilder</h3><ol class="county-ranking">']
    for v in sorted(values.values(),key=lambda v:v['antal_personer']-v['med_fri_commons_bild'],reverse=True):
        missing=v['antal_personer']-v['med_fri_commons_bild']
        parts.append('<li><button class="county-rank" data-name="'+html.escape(v['grupp'],quote=True)+'">'+html.escape(v['grupp'])+'</button> <strong>'+str(missing)+'</strong> <small>('+f'{100*missing/v["antal_personer"]:.1f}'.replace('.',',')+' %)</small></li>')
    parts.append('</ol></div></div><p class="map-source">Förenklade länsgränser: <a href="https://github.com/okfse/sweden-geojson">okfse/sweden-geojson</a>. Kartan visar valkretsarnas län, inte personernas bostadsorter. Saknar fri bild = totalantal minus personer med identifierad fri P18-bild; okänd licens ingår i detta mått.</p></section>')
    return ''.join(parts)

def statistics_html(rows, statistics, geometry):
    free_count = sum(row['fri_commons_bild'] for row in rows)
    share = 100*free_count/len(rows) if rows else 0
    result = '<details class="statistics"><summary>Statistik: '+str(free_count)+' av '+str(len(rows))+' har fri Commons-bild ('+f'{share:.1f}'.replace('.', ',')+' %) — per parti och län</summary>'
    result += '<p>Varje person räknas en gång om minst en P18-bild har en identifierad fri licens i Commons metadata. CC0 och public domain ingår. Okända licenser räknas separat. Uppgifterna avser hela listan och påverkas inte av filtren nedan.</p><p>Län härleds från valkrets, inte från bostadsort. Stockholm stad och län slås ihop; Malmö och Skånes valkretsar till Skåne län; Göteborg och Västra Götalands valkretsar till Västra Götalands län.</p>'
    result += party_chart(statistics)+county_map(statistics,geometry)+'<div class="statistics-tables">'
    for field, title in (('parti', 'Per parti'), ('lan', 'Per län')):
        result += '<section><h2>'+title+'</h2><div class="table-scroll"><table><thead><tr><th scope="col">'+('Parti' if field=='parti' else 'Län')+'</th><th scope="col">Personer</th><th scope="col">Med fri bild</th><th scope="col">Utan bild</th><th scope="col">Licens okänd</th><th scope="col">Andel</th></tr></thead><tbody>'
        for group in statistics[field]:
            result += '<tr><th scope="row">'+html.escape(group['grupp'])+'</th>'+''.join('<td>'+str(group[k])+'</td>' for k in ('antal_personer', 'med_fri_commons_bild', 'utan_commons_bild', 'licens_okand'))+'<td>'+f"{group['andel_procent']:.1f}".replace('.', ',')+' %</td></tr>'
        result += '</tbody><tfoot><tr><th scope="row">Totalt</th><td>'+str(len(rows))+'</td><td>'+str(free_count)+'</td><td>'+str(sum(not row['commons_bilder'] for row in rows))+'</td><td>'+str(sum(bool(row['commons_bilder']) and not row['fri_commons_bild'] for row in rows))+'</td><td>'+f'{share:.1f}'.replace('.', ',')+' %</td></tr></tfoot></table></div></section>'
    return result+'</div></details>'

def illustration_status(rows, out, refresh=False):
    """Kontrollera endast identifierarmatchade sv-artiklar för personer utan P18."""
    target = 'Mall:Illustrationsbehov-svensk politiker'
    titles = sorted({v['titel'] for r in rows if r['qid'] and not r['bilder']
                     for v in r['wikipedia'] if v['sprak'] == 'sv'})
    checked = {}
    def normalize(title):
        title = title.replace('_', ' ').strip()
        return title[:1].upper()+title[1:]
    if titles:
        try:
            aliases = fetch('https://sv.wikipedia.org/w/api.php', out/'svwiki_template_aliases.json', refresh,
                            {'action':'query','format':'json','formatversion':2,'titles':target,
                             'prop':'redirects','rdnamespace':10,'rdlimit':'max'})
            pages = aliases['query']['pages']
            if any(page.get('missing') for page in pages):
                raise ValueError('Illustrationsbehovsmallen kunde inte verifieras.')
            template_titles = {target} | {r['title'] for page in pages for r in page.get('redirects', [])}
            if 'continue' in aliases:
                raise ValueError('Mallens omdirigeringar kräver fortsatt hämtning.')
            for start in range(0, len(titles), 25):
                batch = titles[start:start+25]
                cache = out/('svwiki_illustration_'+hashlib.sha256('|'.join(batch).encode()).hexdigest()[:16]+'.json')
                try:
                    data = fetch('https://sv.wikipedia.org/w/api.php', cache, refresh,
                                 {'action':'query','format':'json','formatversion':2,'titles':'|'.join(batch),
                                  'redirects':1,'prop':'templates','tltemplates':'|'.join(sorted(template_titles)),
                                  'tllimit':'max','curtimestamp':1})
                    if 'continue' in data:
                        raise ValueError('Ofullständigt mallsvar; status lämnas okänd.')
                    resolved = {normalize(v['from']):normalize(v['to']) for key in ('normalized','redirects')
                                for v in data['query'].get(key, [])}
                    page_index = {normalize(v['title']):v for v in data['query']['pages']}
                    for title in batch:
                        canonical=normalize(title);seen=set()
                        while canonical in resolved and canonical not in seen:
                            seen.add(canonical);canonical=resolved[canonical]
                        page=page_index.get(canonical)
                        if page and not page.get('missing') and not page.get('invalid'):
                            checked[title] = {'status':'finns' if page.get('templates') else 'saknas',
                                              'artikel':page['title'], 'kontrollerad':data.get('curtimestamp','')}
                except Exception as error:
                    print('Wikipedia-mallkontroll misslyckades för en batch: '+str(error), flush=True)
        except Exception as error:
            print('Wikipedia-mallkontroll misslyckades: '+str(error), flush=True)
    for r in rows:
        sv = next((v for v in r['wikipedia'] if v['sprak']=='sv'), None)
        applicable = bool(r['qid'] and not r['bilder'] and sv)
        result = checked.get(sv['titel'], {}) if applicable else {}
        r['illustrationsbehov_status'] = result.get('status','okant') if applicable else 'ej_aktuell'
        r['illustrationsbehov_kontrollerad'] = result.get('kontrollerad','')
        r['illustrationsbehov_artikel'] = result.get('artikel','')

def run(directory='.', refresh=False):
    out = Path(directory)
    out.mkdir(parents=True, exist_ok=True)
    raw = fetch(SOURCE, out/'riksdagen_personlista.json', refresh)
    people = raw['personlista']['person']
    if isinstance(people, dict):
        people = [people]
    assert people and len(people) == int(raw['personlista']['@hits'])
    assert len({p['sourceid'].lower() for p in people}) == len(people), 'Dubbletter i Riksdagens källa'
    guid_index = index(sparql(Q8388, out/'wikidata_p8388.json', refresh))
    old_index = index(sparql(Q1214, out/'wikidata_p1214.json', refresh))
    unmatched = [p for p in people if p['sourceid'].lower() not in guid_index]
    values = ' '.join(json.dumps(p['tilltalsnamn']+' '+p['efternamn'], ensure_ascii=False)+'@'+lang
                      for p in unmatched for lang in ('sv', 'en'))
    candidates = []
    if values:
        query = ('SELECT DISTINCT ?item ?label ?description ?birth ?image WHERE { VALUES ?label { '+values+
                 ' } ?item rdfs:label ?label . OPTIONAL { ?item schema:description ?description . FILTER(LANG(?description)="sv") }'
                 ' OPTIONAL { ?item wdt:P569 ?birth } OPTIONAL { ?item wdt:P18 ?image } }')
        candidates = sparql(query, out/'wikidata_name_candidates.json', refresh)
    candidate_index = defaultdict(dict)
    for c in candidates:
        qid = c['item']['value'].rsplit('/', 1)[-1]
        candidate_index[c['label']['value']][qid] = {
            'qid': qid, 'description': c.get('description', {}).get('value', ''),
            'birth': c.get('birth', {}).get('value', '')[:4]}
    rows = []
    for p in people:
        name = p['tilltalsnamn']+' '+p['efternamn']
        matches = guid_index.get(p['sourceid'].lower(), {})
        fallback = old_index.get(p['intressent_id'], {}) if not matches else {}
        fallback_images = set()
        for qid in fallback:
            # Direkt kontroll av P18 för personer hittade via det äldre ID:t.
            entity = fetch('https://www.wikidata.org/wiki/Special:EntityData/'+qid+'.json',
                           out/(qid+'_entity.json'), refresh)['entities'][qid]
            for claim in entity.get('claims', {}).get('P18', []):
                if claim.get('rank') != 'deprecated' and claim['mainsnak'].get('snaktype') == 'value':
                    fallback_images.add(claim['mainsnak']['datavalue']['value'])
        images = sorted({i for img in matches.values() for i in img})
        if len(matches) > 1:
            status = 'Flera P8388-matchningar'
        elif matches:
            status = 'Har bild (P18)' if images else 'Saknar bild (P18)'
        elif fallback:
            status = 'Hittad via P1214 – saknar P8388'
        else:
            status = 'Ingen identifierarmatchning – kontrollera Wikidata'
        rows.append({'namn': name, 'parti': p['parti'], 'valkrets': p['valkrets'],
                     'riksdagsstatus': p['status'], 'fodd_ar': p['fodd_ar'], 'sourceid': p['sourceid'],
                     'intressent_id': p['intressent_id'], 'qid': '; '.join(sorted(matches or fallback)),
                     'status': status, 'bilder': '; '.join(images),
                     'p1214_bildstatus': ('Har P18' if fallback_images else 'Saknar P18') if fallback else '',
                     'riksdagen_bild': p.get('bild_url_192', ''),
                     'kandidater': list(candidate_index[name].values()) if not matches else [],
                     'epost': official_email(p), 'mejl_amne': EMAIL_SUBJECT,
                     'mejl_text': ''})
    # Sitelänkar för identifierarmatchade objekt och tydligt märkta namnkandidater.
    qids = sorted({qid for r in rows for qid in r['qid'].split('; ') if qid} |
                  {c['qid'] for r in rows for c in r['kandidater']})
    wiki_index = defaultdict(dict)
    if qids:
        query = ('SELECT ?item ?article ?site ?title WHERE { VALUES ?item { '+
                 ' '.join('wd:'+qid for qid in qids)+
                 ' } ?article schema:about ?item ; schema:isPartOf ?site ; schema:name ?title . '
                 'FILTER(REGEX(STR(?site), "^https://[a-z-]+[.]wikipedia[.]org/$")) }')
        for b in sparql(query, out/'wikidata_wikipedia.json', refresh):
            qid = b['item']['value'].rsplit('/', 1)[-1]
            language = b['site']['value'].split('://', 1)[1].split('.', 1)[0]
            wiki_index[qid][language] = {'sprak': language, 'titel': b['title']['value'],
                                         'url': b['article']['value']}
    for r in rows:
        links = {lang: value for qid in r['qid'].split('; ') for lang, value in wiki_index[qid].items()}
        r['wikipedia'] = sorted(links.values(), key=lambda v: (v['sprak'] != 'sv', v['sprak'] != 'en', v['sprak']))
        r['mejl_text'] = email_body(r['namn'], r['wikipedia'])
        r['sv_wikipedia'] = ('ja' if 'sv' in links else 'nej') if r['qid'] else 'okant'
        for c in r['kandidater']:
            c['wikipedia'] = sorted(wiki_index[c['qid']].values(), key=lambda v: (v['sprak'] != 'sv', v['sprak']))
    illustration_status(rows, out, refresh)
    image_urls = sorted({url for r in rows for url in r['bilder'].split('; ') if url})
    image_metadata = commons_metadata(image_urls, out, refresh)
    for r in rows:
        r['commons_bilder'] = [image_metadata[unquote(url.rsplit('/', 1)[-1]).replace('_', ' ')]
                               for url in r['bilder'].split('; ') if url]
        r['lan'] = county_from_constituency(r['valkrets'])
        r['fri_commons_bild'] = any(is_free_commons_license(image) for image in r['commons_bilder'])
    rows.sort(key=lambda r: (r['parti'], r['namn']))
    counts = Counter(r['status'] for r in rows)
    assert sum(counts.values()) == len(people)
    stamp = datetime.now(timezone.utc).isoformat()
    summary = {'rapport_genererad_utc': stamp, 'riksdagen_systemdatum': raw['personlista']['@systemdatum'],
               'antal_personer': len(rows), 'status': dict(counts), 'live_refresh': refresh}
    summary['commons_statistik'] = image_statistics(rows)
    # Statiska, förenklade gränser sparas lokalt och i notebookens snapshot.
    geometry = fetch('https://raw.githubusercontent.com/okfse/sweden-geojson/master/swedish_regions.geojson',
                     out/'sweden_counties.geojson', refresh=False)
    (out/'commons_statistik.json').write_text(json.dumps(summary['commons_statistik'], ensure_ascii=False, indent=2), encoding='utf-8')
    for field, table in summary['commons_statistik'].items():
        with (out/('commons_statistik_per_'+field+'.csv')).open('w', encoding='utf-8-sig', newline='') as f:
            writer = csv.DictWriter(f, fieldnames=list(table[0]))
            writer.writeheader()
            writer.writerows(table)
    (out/'sammanfattning.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
    (out/'bildkontroll_resultat.json').write_text(json.dumps(rows, ensure_ascii=False, indent=2), encoding='utf-8')
    fields = [k for k in rows[0] if k not in ('kandidater', 'wikipedia', 'commons_bilder')]
    fields += ['wikipedia_lankar', 'commons_licens', 'commons_fotograf', 'commons_kallor']
    with (out/'bildkontroll_resultat.csv').open('w', encoding='utf-8-sig', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=fields, extrasaction='ignore')
        writer.writeheader()
        writer.writerows(dict(r, wikipedia_lankar='; '.join(v['sprak']+': '+v['url'] for v in r['wikipedia']),
                              commons_licens='; '.join(v['licens'] for v in r['commons_bilder']),
                              commons_fotograf='; '.join(v['fotograf_upphovsperson'] for v in r['commons_bilder']),
                              commons_kallor='; '.join(v['commons_url'] for v in r['commons_bilder'])) for r in rows)
    render_html(rows, summary, out/'riksdagen_saknar_bild.html', geometry)
    return rows, summary

def render_html(rows, summary, path, geometry):
    e = html.escape
    def link(url, label):
        return '<a target="_blank" rel="noopener" href="'+e(url, quote=True)+'">'+e(label)+'</a>'
    cards = []
    for r in rows:
        source = 'https://data.riksdagen.se/personlista/?'+urlencode({'iid': r['sourceid'], 'utformat': 'html'})
        wdlinks = ' · '.join(link('https://www.wikidata.org/wiki/'+qid, qid) for qid in r['qid'].split('; ') if qid)
        search = link('https://www.wikidata.org/w/index.php?'+urlencode({'search': r['namn'], 'title': 'Special:Search', 'ns0': '1'}), 'Sök i Wikidata')
        candidate_links = '<br>'.join(link('https://www.wikidata.org/wiki/'+c['qid'], c['qid'])+' '+
                                      e(c['description'])+' (född '+e(c['birth'] or '?')+')'+
                                      ('<br>Wikipedia för kandidaten: '+ ' · '.join(link(v['url'], v['sprak']) for v in c['wikipedia']) if c['wikipedia'] else '') for c in r['kandidater'])
        wikipedia_links = ' · '.join(link(v['url'], v['sprak']+': '+v['titel']) for v in r['wikipedia'])
        wikipedia_status = {'ja': 'Finns på svenska Wikipedia', 'nej': 'Saknar svensk Wikipedia-artikel',
                            'okant': 'Svensk Wikipedia: okänt – personen är inte identifierarmatchad'}[r['sv_wikipedia']]
        photo = ('<img loading="lazy" alt="Riksdagens porträtt av '+e(r['namn'], quote=True)+'" src="'+e(r['riksdagen_bild'], quote=True)+'">') if r['riksdagen_bild'] else ''
        commons_photos = ''
        for v in r['commons_bilder']:
            preview = ('<img loading="lazy" alt="Commons-bild av '+e(r['namn'], quote=True)+'" src="'+e(v['bild_url'], quote=True)+'">') if v['bild_url'] else ''
            license_url = v['licens_url']
            license_label = link(license_url, v['licens']) if license_url.startswith(('https://', 'http://')) else e(v['licens'])
            commons_photos += '<section class="commons-image">'+preview+'<div><strong>Wikidata P18 / Commons</strong><br>Licens: '+license_label+'<br>Fotograf/upphovsperson: '+e(v['fotograf_upphovsperson'])+'<br>'+link(v['commons_url'], 'Bildens källsida på Commons')+('<br>Credit: '+e(v['credit']) if v['credit'] else '')+('<br>Attribution: '+e(v['attribution']) if v['attribution'] else '')+('<br>Restriktioner: '+e(v['restriktioner']) if v['restriktioner'] else '')+'</div></section>'
        email_section = '<p>Officiell e-postadress saknas i källan.</p>'
        if r['epost']:
            mailto = 'mailto:'+r['epost']+'?'+urlencode({'subject':r['mejl_amne'], 'body':r['mejl_text']}, quote_via=quote)
            email_section = '<section class="email"><p>E-post: '+link('mailto:'+r['epost'], r['epost'])+'</p><p>'+link(mailto, 'Öppna färdigt mejl om porträttbild')+'</p><details><summary>Visa och kopiera mejltexten</summary><p><strong>Ämne: '+e(r['mejl_amne'])+'</strong></p><textarea readonly aria-label="Mejltext till '+e(r['namn'],quote=True)+'">'+e(r['mejl_text'])+'</textarea><button type="button" class="copy-email">Kopiera mejltext</button><span class="copy-status" role="status"></span></details></section>'
        cards.append('<article data-status="'+e(r['status'], quote=True)+'" data-party="'+e(r['parti'], quote=True)+'" data-sv-wikipedia="'+r['sv_wikipedia']+'" data-illustration="'+r['illustrationsbehov_status']+'">'+photo+
                     '<div><span class="badge">'+e(r['status'])+'</span><h2>'+e(r['namn'])+' <small>'+e(r['parti'])+'</small></h2>'+
                     '<p>'+e(r['valkrets'])+' · Född '+e(r['fodd_ar'])+'<br>'+e(r['riksdagsstatus'])+'</p>'+
                     '<p>'+link(source, 'Riksdagen')+' · '+search+(' · '+wdlinks if wdlinks else '')+'</p>'+
                     '<p><strong>'+e(wikipedia_status)+'</strong></p>'+
                     ('<p>Wikipedia: '+wikipedia_links+'</p>' if wikipedia_links else '')+
                     ('<p>Illustrationsbehov-svensk politiker: <strong>'+e({'saknas':'Mallen saknas', 'finns':'Mallen finns', 'okant':'Okänd status'}[r['illustrationsbehov_status']])+'</strong>'+('<br><small>Kontrollerad: '+e(r['illustrationsbehov_kontrollerad'])+'</small>' if r['illustrationsbehov_kontrollerad'] else '')+'</p>' if r['illustrationsbehov_status']!='ej_aktuell' else '')+
                     commons_photos+
                     email_section+
                     '<p class="id">sourceid / P8388: '+e(r['sourceid'])+'<br>intressent_id / P1214: '+e(r['intressent_id'])+'</p>'+
                     ('<p>Bildkontroll av P1214-objekt: '+e(r['p1214_bildstatus'])+'</p>' if r['p1214_bildstatus'] else '')+
                     ('<details><summary>Namnkandidater – kräver identitetskontroll</summary>'+candidate_links+'</details>' if candidate_links else '')+'</div></article>')
    options = ''.join('<option>'+e(s)+'</option>' for s in summary['status'])
    parties = ''.join('<option>'+e(s)+'</option>' for s in sorted({r['parti'] for r in rows}))
    stats = ''.join('<div><strong>'+str(n)+'</strong><span>'+e(s)+'</span></div>' for s, n in summary['status'].items())
    document = '''<!doctype html><html lang="sv"><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">
<title>Riksdagen: bildluckor och Wikidata-matchning</title><style>
body{margin:0;background:#f3f6f9;color:#162538;font:16px/1.5 system-ui,sans-serif}main{max-width:1160px;margin:auto;padding:32px 24px}h1{font-size:34px;line-height:1.15}a{color:#075da6}small{font-size:16px;color:#536173}.stats{display:flex;flex-wrap:wrap;gap:12px;margin:24px 0}.stats div{background:#fff;padding:18px;border-radius:10px;flex:1;min-width:160px}.stats strong{display:block;font-size:32px}.stats span{font-size:14px}.note{background:#e6eef7;border-left:4px solid #376caa;padding:16px}nav{display:flex;flex-wrap:wrap;gap:10px;margin:24px 0}input,select{font:inherit;padding:10px;border:1px solid #aebbc9;border-radius:6px}input{flex:1;min-width:220px}#cards{display:grid;grid-template-columns:repeat(auto-fit,minmax(380px,1fr));gap:16px}article{background:#fff;border:1px solid #dce3eb;border-radius:10px;padding:18px;display:flex;gap:18px}article[hidden]{display:none}img{width:96px;height:128px;object-fit:cover;border-radius:5px}h2{font-size:21px;margin:12px 0 4px}p{margin:9px 0}.badge{font-size:12px;background:#eef1f6;padding:4px 8px;border-radius:5px}.id{font-size:12px;overflow-wrap:anywhere;color:#536173}details{font-size:14px}footer{margin:32px 0;color:#536173;font-size:13px}@media(max-width:500px){#cards{grid-template-columns:1fr}img{width:64px;height:86px}main{padding:18px}article{gap:12px}}@media print{nav{display:none}article{break-inside:avoid}}
.commons-image{display:flex;gap:12px;border-top:1px solid #dce3eb;margin-top:14px;padding-top:14px;font-size:13px}.commons-image img{width:72px;height:96px}.commons-image div{overflow-wrap:anywhere}.email{border-top:1px solid #dce3eb;margin-top:14px;padding-top:8px}.email textarea{display:block;width:100%;box-sizing:border-box;min-height:240px;font:13px/1.5 system-ui;margin:10px 0}.email button{padding:8px;font:inherit;cursor:pointer}.copy-status{margin-left:8px;font-size:13px}.statistics{background:#fff;border:1px solid #dce3eb;border-radius:10px;margin:22px 0;padding:18px}.statistics>summary{font-size:18px;font-weight:650;cursor:pointer}.statistics-tables{display:grid;grid-template-columns:repeat(auto-fit,minmax(460px,1fr));gap:24px}.table-scroll{overflow-x:auto}table{border-collapse:collapse;width:100%;font-size:13px}th,td{padding:9px 6px;border-bottom:1px solid #dce3eb;text-align:right}th:first-child{text-align:left}thead{background:#edf3fa}tfoot{font-weight:700}tbody tr:nth-child(even){background:#f8fafc}@media(max-width:550px){.statistics-tables{grid-template-columns:1fr}}.party-chart{display:block;width:100%;max-width:760px;margin:8px auto}.party-visual,.county-visual{margin:24px 0;padding:18px;background:#f8fafc;border-radius:10px}.chart-legend{display:flex;gap:18px;flex-wrap:wrap}.county-map-layout{display:grid;grid-template-columns:minmax(240px,380px) minmax(240px,1fr);gap:24px;align-items:start}#county-map{width:100%;max-height:640px;display:block;background:#eef5fa;border-radius:10px}.county-path{cursor:pointer;transition:fill .15s}.county-path:hover,.county-path:focus,.county-path.selected{stroke:#15283f;stroke-width:2.2;outline:none}.county-ranking{padding-left:22px;font-size:14px}.county-ranking li{padding:5px 0}.county-rank{border:0;background:none;color:#075da6;text-decoration:underline;cursor:pointer;font:inherit;padding:0}.map-scale{display:flex;align-items:center;gap:10px;margin-top:10px}.map-scale div{height:14px;flex:1;background:linear-gradient(to right,#fff1e7,#aa321f)}.map-source{font-size:12px;color:#536173}#county-selection{padding:14px;background:#e5edf7;min-height:65px;border-radius:6px}@media(max-width:700px){.county-map-layout{grid-template-columns:1fr}#county-map{max-height:520px}}</style><main><h1>Riksdagen → Wikidata<br>Bildluckor och saknade kopplingar</h1>'''
    document += '<p>Källans systemdatum: <strong>'+e(summary['riksdagen_systemdatum'])+'</strong>. '+str(summary['antal_personer'])+' tjänstgörande personer i Riksdagens lista, inklusive ersättare. Rapport genererad '+e(summary['rapport_genererad_utc'])+'.</p>'
    document += statistics_html(rows, summary['commons_statistik'], geometry)
    document += '<section class="stats">'+stats+'</section><div class="note"><strong>Så läses resultatet</strong><br>sourceid matchas mot P8388. P18 kontrolleras i Wikidatas SPARQL-index (icke föråldrade truthy-värden). En utebliven identifierarmatchning bevisar inte att personen saknas i Wikidata. P1214 är en extra kontroll; namnkandidater med exakt svensk eller engelsk etikett kräver manuell kontroll av identitet, födelseår och politisk roll. Svenska Wikipedia-filtret gäller identifierarmatchade objekt; omatchade personer får status okänt. Kandidaternas Wikipedia-länkar ligger under namnkandidater. Indexet kan släpa efter senaste ändringar.<br><br>Porträtten är länkade från Riksdagens bildarkiv och behöver internet. Mejllänkarna öppnar ditt e-postprogram med ett utkast; du granskar och skickar själv. Avsändarnamnet ändras via EMAIL_SENDER i notebooken. Rapporten fastställer ingen bildlicens och föreslår ingen automatisk uppladdning till Commons.</div>'
    document += '<nav><input id="search" placeholder="Sök namn, valkrets, Q-id eller identifierare" aria-label="Sök"><select id="status" aria-label="Status"><option value="">Alla bildstatusar</option>'+options+'</select><select id="party" aria-label="Parti"><option value="">Alla partier</option>'+parties+'</select><select id="svwiki" aria-label="Svenska Wikipedia"><option value="">Svenska Wikipedia: alla</option><option value="ja">Finns på svenska Wikipedia</option><option value="nej">Saknar svensk Wikipedia-artikel</option><option value="okant">Okänt – ej identifierarmatchad</option></select><select id="illustration" aria-label="Illustrationsbehovsmall"><option value="">Illustrationsbehovsmall: alla</option><option value="saknas">Saknad mall (utan P18)</option></select></nav><p class="note">Filtervalen sparas i URL:en. Kopiera webbadressen för att dela samma urval.</p><p id="visible"></p><section id="cards">'+''.join(cards)+'</section>'
    document += '<footer>Källor: '+link(SOURCE,'Riksdagens personlista')+' · '+link('https://www.wikidata.org/wiki/Property:P8388','P8388')+' · '+link('https://www.wikidata.org/wiki/Property:P1214','P1214')+' · '+link('https://www.wikidata.org/wiki/Property:P18','P18')+'. Alla personer visas. Licens och upphovsperson kommer från Commons filmetadata; uppladdaren används inte som ersättning för fotograf. P18-bilden kan skilja sig från den bild som visas i en Wikipedia-artikel.</footer></main>'
    document += '''<script>const cards=[...document.querySelectorAll('article')];const search=document.getElementById('search'),status=document.getElementById('status'),party=document.getElementById('party'),svwiki=document.getElementById('svwiki'),illustration=document.getElementById('illustration');function filter(){let n=0;const q=search.value.toLocaleLowerCase('sv');for(const c of cards){const show=c.textContent.toLocaleLowerCase('sv').includes(q)&&(!status.value||c.dataset.status===status.value)&&(!party.value||c.dataset.party===party.value)&&(!svwiki.value||c.dataset.svWikipedia===svwiki.value)&&(!illustration.value||c.dataset.illustration===illustration.value);c.hidden=!show;if(show)n++;}document.getElementById('visible').textContent=n+' av '+cards.length+' personer visas';}const urlFilters={q:search,status,party,svwiki,illustration};function restoreFilters(){const params=new URLSearchParams(window.location.hash.slice(1));for(const [key,control] of Object.entries(urlFilters)){const value=params.get(key)||'';control.value=control.tagName==='SELECT'&&!Array.from(control.options).some(o=>o.value===value)?'':value;}filter();}function updateFilterUrl(){const params=new URLSearchParams();for(const [key,control] of Object.entries(urlFilters)){if(control.value)params.set(key,control.value);}const url=new URL(window.location.href);url.hash=params.toString();try{window.history.replaceState(null,'',url.href);}catch(error){window.location.hash=params.toString();}}for(const control of Object.values(urlFilters))control.addEventListener('input',()=>{filter();updateFilterUrl();});window.addEventListener('hashchange',restoreFilters);window.addEventListener('popstate',restoreFilters);restoreFilters();document.querySelectorAll('.copy-email').forEach(button=>button.addEventListener('click',async()=>{const box=button.closest('details').querySelector('textarea'),message=button.nextElementSibling;try{if(navigator.clipboard&&window.isSecureContext){await navigator.clipboard.writeText(box.value);}else{box.focus();box.select();if(!document.execCommand('copy'))throw new Error('copy');}message.textContent='Mejltext kopierad';}catch(error){box.focus();box.select();message.textContent='Tryck Ctrl+C eller ⌘C för att kopiera';}}));const mapPaths=[...document.querySelectorAll('.county-path')],metric=document.getElementById('map-metric');function recolor(){const percentage=metric.value==='percent';const max=percentage?100:Math.max(...mapPaths.map(p=>Number(p.dataset.count)));for(const p of mapPaths){const v=Number(percentage?p.dataset.percent:p.dataset.count),t=max?v/max:0;const c=[255,241,231].map((start,i)=>Math.round(start+([170,50,31][i]-start)*t));p.style.fill='rgb('+c.join(',')+')';}document.getElementById('map-max').textContent=percentage?'100 %':max;document.getElementById('map-caption').textContent=percentage?'Andel personer som saknar fri Commons-bild':'Antal personer som saknar fri Commons-bild';}function selectCounty(p){for(const other of mapPaths)other.classList.toggle('selected',other===p);document.getElementById('county-selection').textContent=p.dataset.label;}for(const p of mapPaths){p.addEventListener('click',()=>selectCounty(p));p.addEventListener('keydown',e=>{if(e.key==='Enter'||e.key===' '){e.preventDefault();selectCounty(p);}});}document.querySelectorAll('.county-rank').forEach(b=>b.addEventListener('click',()=>{const p=mapPaths.find(p=>p.dataset.name===b.dataset.name);if(p)selectCounty(p);}));metric.addEventListener('change',recolor);recolor();</script></html>'''
    path.write_text(document, encoding='utf-8')



In [3]:
rows, summary = run(OUTPUT_DIR, refresh=REFRESH)
print(json.dumps(summary, ensure_ascii=False, indent=2))

Hämtar riksdagen_personlista.json …
Hämtar wikidata_p8388.json …
Väntar för att respektera Wikidatas frågegräns (63 sekunder).
Hämtar wikidata_p1214.json …
Väntar för att respektera Wikidatas frågegräns (62 sekunder).
Hämtar wikidata_wikipedia.json …
Hämtar svwiki_template_aliases.json …
Hämtar svwiki_illustration_f5f1361261690d02.json …
Hämtar svwiki_illustration_cbe0d1cfe76bb5bf.json …
Hämtar svwiki_illustration_96d5accabba18df5.json …
Hämtar svwiki_illustration_7d3164d505c494bb.json …
Hämtar svwiki_illustration_037c27aa485857a1.json …
Hämtar svwiki_illustration_1a43ce24bf559ffa.json …
Hämtar svwiki_illustration_4b8a25e885a5e720.json …
Hämtar commons_batch_ecc40b2c20256eb9.json …
Hämtar commons_batch_0ff5d7d77fdf8ba2.json …
Hämtar commons_batch_7a4d7b11d8e0ba5c.json …
Hämtar commons_batch_d47d06d7be67e953.json …
Hämtar commons_batch_7c4d89fbab0e2d9e.json …
Hämtar commons_batch_94e77ada12cfc013.json …
Hämtar commons_batch_7470e6a231aa3e71.json …
Hämtar commons_batch_a722e9a931cd4775.j

In [4]:
from IPython.display import display, FileLink
from collections import Counter
display(FileLink(str(OUTPUT_DIR / 'riksdagen_saknar_bild.html')))
display(FileLink(str(OUTPUT_DIR / 'bildkontroll_resultat.csv')))
print('Svenska Wikipedia bland identifierarmatchade och omatchade:', dict(Counter(r['sv_wikipedia'] for r in rows)))


/Users/salgo/Documents/GitHub/Wikidata_riksdagen-corpus/Notebook/riksdagen_audit_20261008_v10/riksdagen_saknar_bild.html

/Users/salgo/Documents/GitHub/Wikidata_riksdagen-corpus/Notebook/riksdagen_audit_20261008_v10/bildkontroll_resultat.csv

Svenska Wikipedia bland identifierarmatchade och omatchade: {'ja': 329, 'nej': 20}


## Tolkning

- **Saknar bild (P18):** matchad via P8388, inget truthy-bildvärde i Wikidata-indexet.
- **Hittad via P1214 – saknar P8388:** finns via ett annat stabilt person-ID; P18 kontrolleras direkt i objektets JSON.
- **Ingen identifierarmatchning:** kan saknas helt eller finnas utan de kontrollerade identifierarna. Granska födelseår, parti och politisk roll i namnkandidaterna före eventuell nyregistrering. Avsaknad av exakta namnträffar bevisar inte heller att objekt saknas.

Rapportens Riksdagsporträtt är externa bildlänkar. Rapporten anger ingen fastställd bildlicens. HTML-tabellen och filtren fungerar utan nätverk; porträtten kräver internet.

Snapshotfilerna återanvänds när `REFRESH=False`. Efter en uppdatering används de senast hämtade filerna i samma utdatakatalog. Det finns inga automatiska skrivoperationer mot Wikidata eller Commons.